In [3]:
import numpy as np
import matplotlib.pyplot as plt
import dask.dataframe as dd
import pandas as pd

# Get barcode images
import xarray as xr
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import holoviews as hv
from holoviews.operation.datashader import rasterize
import hvplot.xarray # noqa
import panel as pn
import fsspec

pn.extension('tabulator')
# hv.extension('bokeh')
hv.extension('bokeh', 'matplotlib')
# %matplotlib inline

In [4]:
def writedir(directory,overwrite=False):
    if overwrite:
        if os.path.exists(directory):
            shutil.rmtree(directory)
        os.makedirs(directory)
    else:
        if not os.path.exists(directory):
            os.makedirs(directory)

import os
import shutil
import dask
import time
import h5py
import threading
import dask.distributed

from time import sleep
from dask.distributed import Client,progress
from dask.distributed.diagnostics.plugin import WorkerPlugin
from dask_jobqueue import SLURMCluster

# from IPython.core.display import display, HTML
from IPython.display import display, HTML

## Hacky memory trim
import ctypes
import gc

def trim_memory() -> int:
    libc = ctypes.CDLL("libc.so.6")
    return libc.malloc_trim(0)

class dask_controller: #adapted from Charles' code
    def __init__(self,n_workers=6,n_workers_min=6,local=True,queue="short",death_timeout=60.,\
                 walltime='01:00:00',cores=1,processes=1,memory='6GB',\
                 working_directory="./",job_extra_directives=[]):
        self.local = local
        self.n_workers = n_workers
        self.n_workers_min = n_workers_min
        self.walltime = walltime
        self.queue = queue
        self.death_timeout = death_timeout
        self.processes = processes
        self.memory = memory
        self.cores = cores
        self.working_directory = working_directory
        self.job_extra_directives = job_extra_directives

        self.futures = {}

        split_walltime = walltime.split(":")
        wall_hrs,wall_mins,wall_secs = tuple(split_walltime)
        ttl_wall_mins = (60*int(wall_hrs)) + int(wall_mins)

        if ttl_wall_mins < 10:
            raise ValueError("Walltime must be at least 10 mins long!")

        self.lifetime = str(max(ttl_wall_mins-10,10)) + "m"
        print(self.lifetime)
        print(walltime)
        self.worker_extra_args = ["--lifetime", self.lifetime, "--lifetime-stagger", "5m"]

        writedir(working_directory,overwrite=False)

    def startdask(self):
        if self.local:
            self.daskclient = Client()
            self.daskclient.cluster.scale(self.n_workers)
        else:
            self.daskcluster = SLURMCluster(n_workers=self.n_workers_min,queue=self.queue,death_timeout=self.death_timeout,walltime=self.walltime,\
                                   processes=self.processes,memory=self.memory,cores=self.cores,local_directory=self.working_directory,\
                                log_directory=self.working_directory,worker_extra_args=self.worker_extra_args,job_extra_directives=self.job_extra_directives)
            self.daskcluster.adapt(minimum=self.n_workers_min, maximum=self.n_workers,\
                                   interval="1m",wait_count=10)
            self.daskclient = Client(self.daskcluster)

    def shutdown(self, delete_files=True):
        self.reset_worker_memory()
        if not self.local:
            self.daskcluster.close()
        if delete_files:
            for item in os.listdir(self.working_directory):
                if "worker-" in item or "slurm-" in item or ".lock" in item:
                    path = "./" + item
                    if os.path.isfile(path):
                        os.remove(path)
                    elif os.path.isdir(path):
                        shutil.rmtree(path)

    def printprogress(self):
        complete = len([item for item in self.futures if item.status=="finished"])
        print(str(complete) + "/" + str(len(self.futures)))

    def displaydashboard(self):
        link = self.daskcluster.dashboard_link
        display(HTML('<a href="' + link +'">Dashboard</a>'))

    def mapfovs(self,function,fov_list,retries=0):
        self.function = function
        self.retries = retries
        def mapallfovs(fov_number,function=function):
            function(fov_number)
        self.futures = {}
        for fov in fov_list:
            future = self.daskclient.submit(mapallfovs,fov,retries=retries)
            self.futures[fov] = future

    def retry_failed(self):
        self.failed_fovs = [fov for fov,future in self.futures.items() if future.status != 'finished']
        out = self.daskclient.restart()
        self.mapfovs(self.function,self.failed_fovs,retries=self.retries)

    def retry_processing(self):
        self.proc_fovs = [fov for fov,future in self.futures.items() if future.status == 'pending']
        out = self.daskclient.restart()
        self.mapfovs(self.function,self.proc_fovs,retries=self.retries)

    def restart(self):
        self.daskclient.restart()
        with self.failure_lock:
            self.failure_counter_variable.set(0)

    def reset_worker_memory(self):
        self.daskclient.cancel([val for key,val in self.futures.items()])
        self.daskclient.run(gc.collect)
        self.daskclient.run(trim_memory)
        print("Done.")

class hdf5lock:
    def __init__(self,filepath,updateperiod=0.1):
        self.filepath = filepath
        self.lockfile = filepath + ".lock"
        self.updateperiod = updateperiod

    def _lock(self):
        while True:
            if not os.path.exists(self.lockfile):
                open(self.lockfile,'w').close()
                break
            sleep(self.updateperiod)

    def _apply_fn(self,function,iomode,*args,**kwargs):
        try:
            fn_output = function(self.filepath,iomode,*args,**kwargs)
            os.remove(self.lockfile)
            return fn_output
        except:
            os.remove(self.lockfile)
            raise

    def lockedfn(self,function,iomode,*args,**kwargs):
        self._lock()
        fn_output = self._apply_fn(function,iomode,*args,**kwargs)
        return fn_output

def transferjob(sourcedir,targetdir,single_file=False):
    mkdircmd = "mkdir -p '" + targetdir + "'"
    if single_file:
        rsynccmd = "rsync -r '" + sourcedir + "' '" + targetdir + "'"
    else:
        rsynccmd = "rsync -r '" + sourcedir + "/' '" + targetdir + "'"
    wrapcmd = mkdircmd + " && " + rsynccmd
    cmd = "sbatch -p transfer -t 0-12:00 --wrap=\"" + wrapcmd + "\""
    print(cmd)
    os.system(cmd)

In [3]:
import dask
dask.config.set({'distributed.scheduler.active-memory-manager.start': True});
dask.config.set({'distributed.scheduler.worker-ttl': "5m"});
dask.config.set({'distributed.scheduler.allowed-failures': 100});

dask_wd = "/home/lag36/scratch/lag36/dask"

dask_controller_instance = dask_controller(
    walltime="2:00:00",
    local=False,
    n_workers=10,
    n_workers_min=10,#200,
    memory="20GB",
    working_directory=dask_wd,
)
dask_controller_instance.startdask()
dask_controller_instance.displaydashboard()

110m
2:00:00


In [5]:
import pandas as pd

def read_barcode_output_hdf5(filename: str) -> tuple[pd.DataFrame, dict]:
    '''
    Reads the barcode output df (the one after the FISH barcode calling is done),
    and its metadata
    '''
    df_bar_fish = pd.read_hdf(filename) #read in the barcode fish data
    with pd.HDFStore(filename,'r') as store:
        df_bar_fish_metadata = store.get_storer('barcodes').attrs.metadata

    return df_bar_fish, df_bar_fish_metadata

def print_stats(df_bar_fish: pd.DataFrame,
                df_bar_fish_metadata: pd.DataFrame) -> None:
    '''
    Prints some stats from the barcode output df metadata
    '''
    print("Total trenches:           ", df_bar_fish_metadata['Total Trenches'])
    print("Total Trenches With Cells: ", df_bar_fish_metadata['Total Trenches With Cells'])
    print("Removed Bits:             ", df_bar_fish_metadata['Removed Bits'])
    print("Hamming Threshold':      ", df_bar_fish_metadata['Hamming Threshold'])
    print("Called:                   ", len(df_bar_fish))
    print("Called percentage:        ", len(df_bar_fish)/df_bar_fish_metadata['Total Trenches With Cells'])

#### Draw bit histograms

In [6]:
def load_percentiles_dd(filename: str,
                        filename_save_hdf5: str | None,
                        channel_cols: list):
    '''
    Load the percentiles from the parquet files. If load_hdf5 is True, then the file is loaded from the HDF5 file.
        filename (str): the filename of the parquet file
        filename_save_hdf5 (str): the filename to save the HDF5 file to. If None, then the file is not saved.
        channel_cols (list): the list of channel columns to load
    '''    
    df_percentiles = (dd.read_parquet(filename,
                                    engine="pyarrow",
                                    calculate_divisions=True,
                                    columns=['timepoints', 'trenchid'] + channel_cols)
                        .astype({'trenchid':'int32', 'timepoints':'int16'})
                        .compute()
    )
    if filename_save_hdf5 is not None:
        df_percentiles.to_hdf(filename_save_hdf5, key='data')
    return df_percentiles

def load_percentiles_df(filename: str,
                        channel_cols: list) -> pd.DataFrame:
    return pd.read_hdf(filename, usecols=channel_cols)

In [7]:
headpath = '/home/lag36/scratch/lag36/2024-12-05_lLAG10_MBM_Run-2_3-Fiducials_Temp-Fixed_Pipeline-Run-2-2025-05-14/Barcodes/'
filename_percentiles_hdf5 = headpath + 'percentiles.hdf5'
channel_cols=['RFP 98th Percentile', 'Cy5 98th Percentile']
df_percentiles = load_percentiles_df(filename_percentiles_hdf5, channel_cols=channel_cols)

In [16]:
# prefix = '/home/lag36/scratch/lag36/Ecoli//2023-01-02_lDE20_Run_8/Barcodes/'
# prefix = '/home/lag36/scratch/lag36/2024-12-05_lLAG10_MBM_Run-2_3-Fiducials_Temp-Fixed/Barcodes/'
prefix = '/home/lag36/scratch/lag36/2024-08-16_lLAG8_Run-1/Barcodes/'
filename = prefix + '/barcode_output_df.hdf5'
df_bar_fish, df_bar_fish_metadata = read_barcode_output_hdf5(filename)
print_stats(df_bar_fish, df_bar_fish_metadata)

Total trenches:            1075006
Total Trenches With Cells:  612218
Removed Bits:              [1]
Hamming Threshold':       0
Called:                    254438
Called percentage:         0.41560032537429475


In [6]:
prefix = '/home/lag36/scratch/lag36/2025-02-20_lLAG8-MBM-37C-Run-05_3-Fids_Auto-Switch/Barcodes/'

In [14]:
# filename = headpath + '/2022-01-20_lDE20_Final_6/Barcodes/percentiles'
# filename_hdf5 = headpath + '/2022-01-20_lDE20_Final_6/Barcodes/percentiles.hdf5'
filename_percentiles = prefix + 'percentiles_98-2'
# filename_percentiles_hdf5 = prefix + 'percentiles_98-minus-2.hdf5'
filename_percentiles_hdf5 = prefix + 'percentiles_98-2.hdf5'
# df_percentiles = load_percentiles_dd(filename_percentiles,
#                                     filename_save_hdf5=filename_percentiles_hdf5,
#                                     channel_cols=['RFP 98th Percentile', 'Cy5 98th Percentile'])

# df_percentiles = load_percentiles(filename,
#                                   load_hdf5=False,
#                                   filename_save_hdf5=filename_hdf5,
#                                   channel_cols=['RFP 98th Percentile', 'Cy5 98th Percentile', 'Cy7 98th Percentile'])

# df_percentiles = load_percentiles_df(filename_percentiles_hdf5,
#                                      channel_cols=['GFP 98th Percentile', 'Cy5 98th Percentile', 'Cy7 98th Percentile'])

# df_percentiles = load_percentiles_df(filename_percentiles_hdf5,
#                                      channel_cols=['RFP 98th Percentile', 'Cy5 98th Percentile'])


filename_percentiles_hdf5 = prefix + 'percentiles.hdf5'
df_percentiles = load_percentiles_df(filename_percentiles_hdf5,
                                     channel_cols=['RFP 98th Percentile', 'Cy5 98th Percentile'])

filename_percentiles_98_2_hdf5 = prefix + 'percentiles_98-2.hdf5'
df_percentiles_98_2 = load_percentiles_df(filename_percentiles_98_2_hdf5,
                                     channel_cols=['RFP 98th Percentile', 'Cy5 98th Percentile'])

In [14]:
from holoviews.operation import histogram
import panel

In [15]:
def generate_percentile_histograms(df_percentiles: pd.DataFrame, 
                                   sample:int,
                                   channel_cols: list,
                                   bin_ranges: dict,
                                   y_lims: dict,
                                   colors: dict) -> dict:
    '''
    Generates histograms of the percentiles for each channel.
        df_percentiles (pd.DataFrame): the dataframe containing the percentiles
        sample (int): the number of trenches to sample
        channel_cols (list): the list of channel columns to plot (e.g. ['RFP 98th Percentile', 'Cy5 98th Percentile', 'Cy7 98th Percentile'])
        bin_ranges (dict): the bin ranges for each channel(e.g. {'RFP 98th Percentile': (0, 100), 'Cy5 98th Percentile': (0, 100), 'Cy7 98th Percentile': (0, 100)})
        y_lims (dict): the y limits for each channel (e.g. {'RFP 98th Percentile': (0, 100), 'Cy5 98th Percentile': (0, 100), 'Cy7 98th Percentile': (0, 100)})
        colors (dict): the colors for each channel (e.g. {'RFP 98th Percentile': 'red', 'Cy5 98th Percentile': 'green', 'Cy7 98th Percentile': 'blue'})
    Returns:
        dict: a dictionary of holoviews histograms for each channel
    '''
    
    trench_ids = df_percentiles['trenchid'].unique()
    trench_ids = np.random.choice(trench_ids, sample, replace=False)

    df_percentiles_sample = (df_percentiles
                    .loc[df_percentiles['trenchid'].isin(trench_ids)]
                    .reset_index()
                    .loc[:,['timepoints', 'trenchid'] + channel_cols]
                    )

    data_hists = hv.Dataset(df_percentiles_sample)
    hists = {}
    for ch in channel_cols:
        hists[ch] = histogram(
            data_hists.to(hv.Points, kdims=['trenchid'] + [ch], groupby=['timepoints']),
            num_bins = 500,
            bin_range = bin_ranges[ch],
            dimension=ch,
        ).opts(
            ylim = y_lims[ch],
            width = 300,
            height=150,
            # Set bar colors to red
            # color='red',
            # Set fill color to blue
            # fill_color='red',
            # Set edge line color to black
            line_color=colors[ch],
        )
    
    return hists

def generate_bit_threshold_lines(bit_thresh_list: list,
                                 channel_cols: list) -> dict:
    '''
    Generates the bit threshold holoviews VLines for each channel, for a given experiment.
        bit_thresh_list (list): the list of bit thresholds for each channel, in order (ch_1, cyc_1), (ch_1, cyc_2),...(ch_2, cyc_1), (ch_2, cyc_2),...
        channel_cols (list): the list of channel columns to plot (e.g. ['RFP', 'Cy5', 'Cy7'])
    Returns:
        dict: a dictionary of holoviews VLines for each channel
    '''
    assert len(bit_thresh_list) % len(channel_cols) == 0 # make sure the list is divisible by the number of channels (i.e. each channel has the same number of thresholds)
    bits_per_channel = len(bit_thresh_list) // len(channel_cols)

    # Create a dictionary of bit thresholds for each channel
    bit_thresholds = {channel_cols[i]: bit_thresh_list[i*bits_per_channel:(i+1)*bits_per_channel] for i in range(len(channel_cols))}
    bit_thresholds['timepoints'] = np.arange(bits_per_channel, dtype=np.int8)
    bit_thresholds = hv.Dataset(pd.DataFrame(bit_thresholds))

    # Create the VLines for each channel
    lines = {}
    for ch in channel_cols:
        lines[ch] = bit_thresholds.to(hv.VLines, kdims=[ch], groupby=['timepoints']).opts(
            line_width=1,
        )
    return lines

#### B. subtilis lLAG10 lines adjustment

In [16]:
# 2 colors, 15 cycles
bit_thresh_list = bit_thresh_list = [2400, #1 RFP
 2200, #2
 2400, #3
 2400, #4
 2400, #5
 2400, #6
 2400, #7
 2400, #8
 2400, #9
 2400, #10
 2400, #11
 2400, #12
 2400, #13
 2400, #14
 2400, #15
 4000, #1 #Cy5
 4000, #2
 4000, #3
 4000, #4
 4000, #5
 4000, #6
 4000, #7
 4000, #8
 3600, #9
 3600, #10
 3600, #11
 3600,
 3600,
 4000,
 3600]

In [17]:
# lLAG10 From notebook
bit_thresh_list = [2350, #1 RFP
 2350, #2
 2300, #3
 2250, #4
 2250, #5
 2250, #6
 2200, #7
 2180, #8
 2200, #9
 2200, #10
 2180, #11
 2200, #12
 2180, #13
 2170, #14
 2130, #15
 4200, #1 #Cy5
 4200, #2
 4200, #3
 4200, #4
 4100, #5
 4200, #6
 4100, #7
 4100, #8
 3850, #9
 4100, #10
 4100, #11
 4100,
 4100,
 4000,
 4200]

In [ ]:
# 3 colors, 10 cycles
bit_thresh_list = [2000, #1 RFP
 2000, #2
 1900, #3
 1900, #4
 1900, #5
 1800, #6
 1900, #7
 1900, #8
 1800, #9
 1800, #10
 5200, #11
 5200, #12
 5200, #13
 5200, #14
 5200, #15
 5200, #1 #Cy5
 5200, #2
 5200, #3
 5200, #4
 5200, #5
 1350, #6 #Cy7: 1
 1450, #7 #Cy7: 2
 1400, #8 #Cy7: 3
 1400, #9 #Cy7: 4
 1500, #10 #Cy7: 5
 1350, #11 #Cy7: 6
 1350, #12 #Cy7: 7
 1280, #13 #Cy7: 8
 1320, #14 #Cy7: 9
 1280] #15 #Cy7: 10

In [39]:
# 2 colors, 15 cycles
bit_thresh_list = bit_thresh_list = [1000, #1 RFP
 750, #2
 1300, #3
 1300, #4
 1300, #5
 1300, #6
 1300, #7
 1300, #8
 1300, #9
 1300, #10
 1300, #11
 1300, #12
 1000, #13
 1300, #14
 1300, #15
 2000, #1 #Cy5
 2000, #2
 2000, #3
 2000, #4
 2000, #5
 2000, #6
 2000, #7
 2000, #8
 1400, #9
 1400, #10
 2000, #11
 2000,
 2000,
 2000,
 2000]

In [56]:
# # Generate histograms # BSUB
# sample = 200000
# channel_cols=['RFP 98th - 2nd Percentile', 'Cy5 98th - 2nd Percentile']
# # channel_cols = ['RFP 98th Percentile', 'Cy5 98th Percentile', 'Cy7 98th Percentile']
# # channel_cols = ['GFP 98th Percentile', 'Cy5 98th Percentile', 'Cy7 98th Percentile']
# # channel_cols = ['RFP 98th Percentile', 'Cy5 98th Percentile']

# bin_ranges = {}
# # bin_ranges['RFP 98th Percentile'] = (0, 6000)
# bin_ranges['RFP 98th - 2nd Percentile'] = (0, 3000)
# bin_ranges['Cy5 98th - 2nd Percentile'] = (0, 6000)
# # bin_ranges['Cy7 98th Percentile'] = (1200, 2400)

# y_lims = {}
# # y_lims['RFP 98th Percentile'] = (0, 3000)
# y_lims['RFP 98th - 2nd Percentile'] = (0, 300)
# y_lims['Cy5 98th - 2nd Percentile'] = (0, 300)
# # y_lims['Cy7 98th Percentile'] = (0, 1000)

# colors = {}
# colors['RFP 98th - 2nd Percentile'] = 'orange'
# colors['Cy5 98th - 2nd Percentile'] = 'magenta'
# # colors['Cy7 98th Percentile'] = 'red'

# hists = generate_percentile_histograms(df_percentiles,
#                                        sample = 200000,
#                                        channel_cols = channel_cols,
#                                        bin_ranges = bin_ranges,
#                                        y_lims = y_lims,
#                                        colors = colors)

# # Generate lines
# channel_cols=['RFP 98th - 2nd Percentile', 'Cy5 98th - 2nd Percentile']
# # bit_thresh_list = df_bar_fish_metadata['Bit Threshold List']
# lines = generate_bit_threshold_lines(bit_thresh_list, channel_cols)

# # Generate NdLayouts of histograms
# hist_grids = {ch: hv.NdLayout(hists[ch] * lines[ch]).cols(2) for ch in channel_cols}
# # hist_grids = {ch: hv.NdLayout(hists[ch]).cols(2) for ch in channel_cols}

# # Generate super_layout
# super_layout = panel.Row(
#     hist_grids['RFP 98th - 2nd Percentile'],
#     hist_grids['Cy5 98th - 2nd Percentile'],
#     # hist_grids['Cy7 98th Percentile'],
# )

# # hist_grids['Cy5 98th - 2nd Percentile']

In [66]:
# hist_grids['RFP 98th - 2nd Percentile']

In [12]:
# Generate histograms # BSUB
sample = 200000
channel_cols=['RFP 98th Percentile', 'Cy5 98th Percentile']
# channel_cols = ['RFP 98th Percentile', 'Cy5 98th Percentile', 'Cy7 98th Percentile']
# channel_cols = ['GFP 98th Percentile', 'Cy5 98th Percentile', 'Cy7 98th Percentile']
# channel_cols = ['RFP 98th Percentile', 'Cy5 98th Percentile']

bin_ranges = {}
# bin_ranges['RFP 98th Percentile'] = (0, 6000)
bin_ranges['RFP 98th Percentile'] = (0, 6000)
bin_ranges['Cy5 98th Percentile'] = (2500, 8000)
# bin_ranges['Cy7 98th Percentile'] = (1200, 2400)

y_lims = {}
# y_lims['RFP 98th Percentile'] = (0, 3000)
y_lims['RFP 98th Percentile'] = (0, 1000)
y_lims['Cy5 98th Percentile'] = (0, 1000)
# y_lims['Cy7 98th Percentile'] = (0, 1000)

colors = {}
colors['RFP 98th Percentile'] = 'orange'
colors['Cy5 98th Percentile'] = 'magenta'
# colors['Cy7 98th Percentile'] = 'red'

hists = generate_percentile_histograms(df_percentiles_98_2,
                                       sample = 200000,
                                       channel_cols = channel_cols,
                                       bin_ranges = bin_ranges,
                                       y_lims = y_lims,
                                       colors = colors)

# Generate lines
channel_cols = ['RFP 98th Percentile', 'Cy5 98th Percentile']#,'Cy7 98th Percentile']
# bit_thresh_list = df_bar_fish_metadata['Bit Threshold List']
lines = generate_bit_threshold_lines(bit_thresh_list, channel_cols)

# Generate NdLayouts of histograms
# hist_grids = {ch: hv.NdLayout(hists[ch] * lines[ch]).cols(2) for ch in channel_cols}
hist_grids = {ch: hv.NdLayout(hists[ch]).cols(2) for ch in channel_cols}

# Generate super_layout
super_layout = panel.Row(
    hist_grids['RFP 98th Percentile'],
    hist_grids['Cy5 98th Percentile'],
    # hist_grids['Cy7 98th Percentile'],
)

super_layout

NameError: name 'df_percentiles_98_2' is not defined

In [36]:
# lLAG10 From notebook
bit_thresh_list = [2200, #1 RFP
 2150, #2
 2300, #3
 2250, #4
 2250, #5
 2250, #6
 2200, #7
 2180, #8
 2200, #9
 2200, #10
 2180, #11
 2200, #12
 2180, #13
 2170, #14
 2130, #15
 4200, #1 #Cy5
 4200, #2
 4200, #3
 4200, #4
 4100, #5
 4200, #6
 4100, #7
 4100, #8
 3350, #9
 4100, #10
 4100, #11
 4100,
 4100,
 4000,
 4200]

In [38]:
# Generate histograms # BSUB
sample = 200000
channel_cols=['RFP 98th Percentile', 'Cy5 98th Percentile']
# channel_cols = ['RFP 98th Percentile', 'Cy5 98th Percentile', 'Cy7 98th Percentile']
# channel_cols = ['GFP 98th Percentile', 'Cy5 98th Percentile', 'Cy7 98th Percentile']
# channel_cols = ['RFP 98th Percentile', 'Cy5 98th Percentile']

bin_ranges = {}
# bin_ranges['RFP 98th Percentile'] = (0, 6000)
bin_ranges['RFP 98th Percentile'] = (1200, 6000)
bin_ranges['Cy5 98th Percentile'] = (2000, 12000)
# bin_ranges['Cy7 98th Percentile'] = (1200, 2400)

y_lims = {}
# y_lims['RFP 98th Percentile'] = (0, 3000)
y_lims['RFP 98th Percentile'] = (0, 1000)
y_lims['Cy5 98th Percentile'] = (0, 1000)
# y_lims['Cy7 98th Percentile'] = (0, 1000)

colors = {}
colors['RFP 98th Percentile'] = 'orange'
colors['Cy5 98th Percentile'] = 'magenta'
# colors['Cy7 98th Percentile'] = 'red'

hists = generate_percentile_histograms(df_percentiles,
                                       sample = 200000,
                                       channel_cols = channel_cols,
                                       bin_ranges = bin_ranges,
                                       y_lims = y_lims,
                                       colors = colors)

# Generate lines
channel_cols = ['RFP 98th Percentile', 'Cy5 98th Percentile']#,'Cy7 98th Percentile']
# bit_thresh_list = df_bar_fish_metadata['Bit Threshold List']
lines = generate_bit_threshold_lines(bit_thresh_list, channel_cols)

# Generate NdLayouts of histograms
hist_grids = {ch: hv.NdLayout(hists[ch] * lines[ch]).cols(2) for ch in channel_cols}
# hist_grids = {ch: hv.NdLayout(hists[ch]).cols(2) for ch in channel_cols}

# Generate super_layout
# super_layout = panel.Row(
#     hist_grids['RFP 98th Percentile'],
#     hist_grids['Cy5 98th Percentile'],
#     # hist_grids['Cy7 98th Percentile'],
# )

hist_grids['Cy5 98th Percentile']

:NdLayout   [timepoints]
   :Overlay
      .Histogram.I :Histogram   [Cy5 98th Percentile]   (Count)
      .VLines.I    :VLines   [Cy5 98th Percentile]

In [ ]:
# I need to inspect a few trenches with the given cutoffs

#### B. subtilis lLAG8 cutoffs

In [71]:
bit_thresh_list = [1750, #1 RFP
 1750, #2
 2000, #3
 2000, #4
 2200, #5
 2200, #6
 2200, #7
 2200, #8
 2000, #9
 2200, #10
 2000, #11
 2000, #12
 2000, #13
 2000, #14
 2200, #15
 6200, #1 #Cy5
 6200, #2
 6200, #3
 6200, #4
 6200, #5
 6200, #6
 6000, #7
 6000, #8
 6000, #9
 6200, #10
 6000, #11
 6000,
 6000,
 6500,
 6000]

In [72]:
# Generate histograms # BSUB
sample = 200000
# channel_cols = ['RFP 98th Percentile', 'Cy5 98th Percentile', 'Cy7 98th Percentile']
# channel_cols = ['GFP 98th Percentile', 'Cy5 98th Percentile', 'Cy7 98th Percentile']
channel_cols = ['RFP 98th Percentile', 'Cy5 98th Percentile']

bin_ranges = {}
# bin_ranges['RFP 98th Percentile'] = (0, 6000)
bin_ranges['RFP 98th Percentile'] = (1000, 4000)
bin_ranges['Cy5 98th Percentile'] = (5000, 12000)
# bin_ranges['Cy7 98th Percentile'] = (0, 4000)

y_lims = {}
# y_lims['RFP 98th Percentile'] = (0, 3000)
y_lims['RFP 98th Percentile'] = (0, 1000)
y_lims['Cy5 98th Percentile'] = (0, 1000)
# y_lims['Cy7 98th Percentile'] = (0, 3000)

colors = {}
colors['RFP 98th Percentile'] = 'orange'
colors['Cy5 98th Percentile'] = 'magenta'
# colors['Cy7 98th Percentile'] = 'red'

hists = generate_percentile_histograms(df_percentiles,
                                       sample = 200000,
                                       channel_cols = channel_cols,
                                       bin_ranges = bin_ranges,
                                       y_lims = y_lims,
                                       colors = colors)

# Generate lines
channel_cols = ['RFP 98th Percentile', 'Cy5 98th Percentile',]
# bit_thresh_list = df_bar_fish_metadata['Bit Threshold List']
lines = generate_bit_threshold_lines(bit_thresh_list, channel_cols)

# Generate NdLayouts of histograms
hist_grids = {ch: hv.NdLayout(hists[ch] * lines[ch]).cols(2) for ch in channel_cols}
# hist_grids = {ch: hv.NdLayout(hists[ch]).cols(2) for ch in channel_cols}

# Generate super_layout
super_layout = panel.Row(
    hist_grids['RFP 98th Percentile'],
    hist_grids['Cy5 98th Percentile'],
    # hist_grids['Cy7 98th Percentile'],
)

super_layout

BokehModel(combine_events=True, render_bundle={'docs_json': {'f8578045-11a0-4942-9263-71dae46a983c': {'version…

#### Continue with plotting functions

In [43]:
# Generate histograms #ECO
sample = 200000
# channel_cols = ['RFP 98th Percentile', 'Cy5 98th Percentile', 'Cy7 98th Percentile']
# channel_cols = ['GFP 98th Percentile', 'Cy5 98th Percentile', 'Cy7 98th Percentile']
channel_cols = ['RFP 98th Percentile', 'Cy5 98th Percentile']

bin_ranges = {}
# bin_ranges['RFP 98th Percentile'] = (0, 6000)
bin_ranges['GFP 98th Percentile'] = (0, 3000)
bin_ranges['Cy5 98th Percentile'] = (0, 20000)
bin_ranges['Cy7 98th Percentile'] = (0, 4000)

y_lims = {}
# y_lims['RFP 98th Percentile'] = (0, 3000)
y_lims['GFP 98th Percentile'] = (0, 3000)
y_lims['Cy5 98th Percentile'] = (0, 3000)
y_lims['Cy7 98th Percentile'] = (0, 3000)

colors = {}
colors['GFP 98th Percentile'] = 'green'
colors['Cy5 98th Percentile'] = 'magenta'
colors['Cy7 98th Percentile'] = 'red'

hists = generate_percentile_histograms(df_percentiles,
                                       sample = 200000,
                                       channel_cols = channel_cols,
                                       bin_ranges = bin_ranges,
                                       y_lims = y_lims,
                                       colors = colors)

# Generate lines
channel_cols = ['GFP 98th Percentile', 'Cy5 98th Percentile', 'Cy7 98th Percentile']
bit_thresh_list = df_bar_fish_metadata['Bit Threshold List']
lines = generate_bit_threshold_lines(bit_thresh_list, channel_cols)

# Generate NdLayouts of histograms
hist_grids = {ch: hv.NdLayout(hists[ch] * lines[ch]).cols(2) for ch in channel_cols}

# Generate super_layout
super_layout = panel.Row(
    hist_grids['GFP 98th Percentile'],
    hist_grids['Cy5 98th Percentile'],
    hist_grids['Cy7 98th Percentile'],
)

KeyError: 'RFP 98th Percentile'

In [17]:
super_layout

BokehModel(combine_events=True, render_bundle={'docs_json': {'2c62d9f5-53e5-40b2-9f13-a3dcf6e54566': {'version…

#### Single trench images per trenchid and timepoint

In [18]:
# Now include the trench kymos

# Read the kymograph metadata and merge with fov data
def read_dd_barcode_kymograph_metadata_to_hdf(path: str,
                                              cols_bar_kymo_meta: list,
                                              hdf_filename: str) -> None:
    '''
    Reads the barcode kymograph metadata from a parquet file and saves it to an HDF5 file.
      path (str): the path to the parquet file
      cols_bar_kymo_meta (list): the columns to read from the parquet file
      hdf_filename (str): the filename to save the data to the HDF5 file
    '''                                      
    df_bar_kymo_meta =  (dd.read_parquet(path,
                             engine="pyarrow",
                             calculate_divisions=True,
                             columns = cols_bar_kymo_meta
                             )
                             .astype({'fov':'int16', 'timepoints':'int16', 'lane orientation':'category', 'row':'int8',
                                              'File Index':'int32', 'File Trench Index':'int32', 'trenchid': 'int32'})
                             .compute()
                                        
            )
    df_bar_kymo_meta.to_hdf(hdf_filename, key='data', format='table')

def read_df_barcode_kymograph_metadata(filename: str,
                                       cols_bar_kymo_meta: list,
                                       filename_fovs: str | None = None,
                                       cols_bar_fovs_meta: list = None,
                                      ) -> pd.DataFrame:
  '''
  Reads the barcode kymograph metadata from an HDF5 file.
    filename (str): the filename of the HDF5 file
    
  '''
  df_bar_kymo_meta = (pd.read_hdf(filename,
                                        key='data',
                                        )
                            .loc[:, cols_bar_kymo_meta] # Typically dask dataframe but I saved it as hdf5
                            .astype({'fov':'int16', 'timepoints':'int16', 'lane orientation':'category', 'row':'int8',
                                              'File Index':'int32', 'File Trench Index':'int32', 'trenchid': 'int32'})
  )

  # Merge fov metadata (e.g. to draw bounding boxes on the raw images)
  if filename_fovs is not None:
    df_bar_kymo_meta = (df_bar_kymo_meta
                            # Merge Barcode FOV metadata
                            .merge(
                             (pd.read_hdf(filename_fovs)
                              .loc[:, cols_bar_fovs_meta]
                              .astype({'File Index':'int32', 'Image Index':'int32'})
                              # Better way to do this?
                              .reset_index()
                              .astype({'fov':'int16', 'timepoints':'int16'})
                             ),
                             on=['fov', 'timepoints'],
                             how='left',
                             suffixes=('', '_fov') 
                            )
                          )
  return df_bar_kymo_meta

In [19]:
# prefix = '/home/lag36/scratch/lag36/Ecoli//2022-01-20_lDE20_Final_6/Barcodes/'
prefix = '/home/lag36/scratch/lag36/Ecoli//2023-01-02_lDE20_Run_8/Barcodes/'
path = prefix+'/kymograph/metadata'

cols_bar_kymo_meta = ['fov', 'timepoints', 'lane orientation', 'y (local)', 
                      'x (local)', 'row', 'File Index', 'File Trench Index',
                      'trenchid', 'Trenchid Timepoint Index']

cols_bar_fovs_meta = ['t', 'File Index', 'Image Index']

hdf5_filename = prefix + 'kymograph_metadata_df.hdf5'

# Only needed once per dataset
# df_bar_kymo_meta = read_dd_barcode_kymograph_metadata_to_hdf(path,
#                                                              cols_bar_kymo_meta,
#                                                              hdf5_filename)

df_bar_kymo_meta_all = read_df_barcode_kymograph_metadata(hdf5_filename,
                                                    #   prefix + 'metadata_df.hdf5',
                                                      cols_bar_kymo_meta,
                                                    #   cols_bar_fovs_meta
                                                      )

In [20]:
# Filter for trench info only (no timepoints)
df_bar_kymo_meta_trench = (df_bar_kymo_meta_all
                      [['trenchid', 'File Index', 'File Trench Index']]
                      .groupby('trenchid')
                      .first()
                      .reset_index()
                      )

In [21]:
def load_kymo_images(filename: str,
                     df_bar_kymo_meta_trench: pd.DataFrame,
                     file_index = None)-> xr.Dataset:
    '''
    Loads the kymograph images from an HDF5 file into an xarray dataset.
    '''
    # TODO Fix the file_index thing. Make it more general
    data_kymo = (xr
        .open_dataset(filename, phony_dims='sort')
        .swap_dims({'phony_dim_0':'File Trench Index', 'phony_dim_1':'timepoints', 'phony_dim_2':'y', 'phony_dim_3':'x' })
    )

    data_kymo = (data_kymo
             .assign_coords({'File Trench Index': np.arange(data_kymo.sizes['File Trench Index'], dtype=np.int16),
                             'timepoints': np.arange(data_kymo.sizes['timepoints'], dtype=np.int16),
                             'y': np.arange(data_kymo.sizes['y'], dtype=np.int16),
                             'x': np.arange(data_kymo.sizes['x'], dtype=np.int16),
                             })
    )
    if file_index is not None:
        data_kymo = (data_kymo
                             # This assumed ordered trenchids in the kymo file
                             .assign_coords(trenchid= ('File Trench Index', df_bar_kymo_meta_trench.loc[df_bar_kymo_meta_trench['File Index']==file_index, 'trenchid']))
        )
    return data_kymo

In [22]:
prefix = '/home/lag36/scratch/lag36/Ecoli//2023-01-02_lDE20_Run_8/Barcodes/'
file_index = 100
filename = f'{prefix}/kymograph/kymograph_{file_index}.hdf5'

kymo_imgs = load_kymo_images(filename, df_bar_kymo_meta_trench, file_index)

In [23]:
kymo_imgs

<xarray.Dataset> Size: 180MB
Dimensions:            (File Trench Index: 1000, timepoints: 10, y: 150, x: 15)
Coordinates:
  * File Trench Index  (File Trench Index) int16 2kB 0 1 2 3 ... 996 997 998 999
  * timepoints         (timepoints) int16 20B 0 1 2 3 4 5 6 7 8 9
  * y                  (y) int16 300B 0 1 2 3 4 5 6 ... 144 145 146 147 148 149
  * x                  (x) int16 30B 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14
    trenchid           (File Trench Index) int32 4kB 100000 100001 ... 100999
Data variables:
    BF                 (File Trench Index, timepoints, y, x) uint16 45MB ...
    Cy5                (File Trench Index, timepoints, y, x) uint16 45MB ...
    Cy7                (File Trench Index, timepoints, y, x) uint16 45MB ...
    GFP                (File Trench Index, timepoints, y, x) uint16 45MB ...

In [24]:
def plot_bc_trench_per_trenchid_timepoint(kymo_imgs: xr.Dataset, 
                                          channels: list,
                                          groupby_cols) -> dict:
    '''
    Plots the barcode trench images per trenchid and timepoint
        kymo_imgs (xr.Dataset): the barcode kymograph images
        channels (list): the list of channels to plot
    Returns:
        dict: a dictionary of holoviews images for each
    '''
    hv_kymos = {}
    for channel in channels:
        hv_kymos[channel] = (kymo_imgs[channel].hvplot.image(
                                groupby=groupby_cols,
                                title=channel,
                                cmap='gray',
                                data_aspect=1,
        )
        .opts(invert_yaxis=True,
                                     # Remove axes labels and ticks
                                     xaxis=None, yaxis=None,
                                     colorbar=False,))
    return hv_kymos

In [25]:
channels = ['BF', 'GFP', 'Cy5', 'Cy7']
groupby_cols = ['File Trench Index', 'timepoints']
hv_kymos = plot_bc_trench_per_trenchid_timepoint(kymo_imgs,
                                                 channels=channels,
                                                 groupby_cols=groupby_cols)

hv_kymos['BF'] + hv_kymos['GFP'] + hv_kymos['Cy5'] + hv_kymos['Cy7']

BokehModel(combine_events=True, render_bundle={'docs_json': {'100241e1-3eb1-40b7-9978-472b7c5d229f': {'version…

In [26]:
# I need to load df_bar to get the annotations, and the mapped kymos
# I need to show the kymos
# I need to find the cutoffs and include them
# TODO: Find a more modular way to include the plotting paramenters
# TODO: Everything should be agnostic to channel number, channel identity, etc.

#### Kymograph per trenchid

In [27]:
def unfold_xr_kymograph(kymo_imgs):
    '''
    Unfolds the xarray kymograph images into a pandas dataframe
    '''
    # Concatenates each timepoint along the x dimension
    kymo_imgs_unfold = xr.concat([kymo_imgs.isel({'timepoints':i}) for i in range(kymo_imgs.sizes['timepoints'])], dim='x')
    # Reassigns the x coordinate
    kymo_imgs_unfold = kymo_imgs_unfold.assign_coords({'x': np.arange(kymo_imgs_unfold.sizes['x'], dtype=np.int16)})
    return kymo_imgs_unfold

In [28]:
kymo_imgs_unfold = unfold_xr_kymograph(kymo_imgs)

In [29]:
kymo_imgs_unfold

<xarray.Dataset> Size: 180MB
Dimensions:            (File Trench Index: 1000, y: 150, x: 150)
Coordinates:
  * File Trench Index  (File Trench Index) int16 2kB 0 1 2 3 ... 996 997 998 999
    timepoints         (x) int16 300B 0 0 0 0 0 0 0 0 0 0 ... 9 9 9 9 9 9 9 9 9
  * y                  (y) int16 300B 0 1 2 3 4 5 6 ... 144 145 146 147 148 149
    trenchid           (File Trench Index) int32 4kB 100000 100001 ... 100999
  * x                  (x) int16 300B 0 1 2 3 4 5 6 ... 144 145 146 147 148 149
Data variables:
    BF                 (File Trench Index, y, x) uint16 45MB 11062 ... 3332
    Cy5                (File Trench Index, y, x) uint16 45MB 644 662 ... 695 743
    Cy7                (File Trench Index, y, x) uint16 45MB 586 488 ... 731 582
    GFP                (File Trench Index, y, x) uint16 45MB 953 979 ... 711 787

In [30]:
def plot_bc_trenches_single_trenchid_unfold(kymo_imgs_unfold: xr.Dataset, 
                                       trenchid: int,
                                       channels: list) -> dict:
    '''
    Plots the barcode trench images per trenchid and timepoint
        kymo_imgs_unfold (xr.Dataset): the barcode kymograph images
        trenchid (int): the trenchid to plot
        channels (list): the list of channels to plot
    Returns:
        dict: a dictionary of holoviews images for each
    '''
    hv_kymos = {}
    for channel in channels:
        hv_kymos[channel] = \
            (kymo_imgs_unfold[channel]
             .where(kymo_imgs_unfold['trenchid'] == trenchid, drop=True)
             .hvplot.image(
                 x='x',
                 y='y',
                 cmap='gray',
                 data_aspect=1,
                 title=f'{channel} {trenchid}'
                 )
             .opts(invert_yaxis=False,
                   xaxis=None,
                   yaxis=None,
                   colorbar=False
                   )
            )
    return hv_kymos

In [31]:
trenchid=100005
channels = ['BF', 'GFP', 'Cy5', 'Cy7']
hv_kymos = plot_bc_trenches_single_trenchid_unfold(kymo_imgs_unfold, trenchid=trenchid, channels=channels)
hv_kymos['BF'] + hv_kymos['GFP'] + hv_kymos['Cy5'] + hv_kymos['Cy7']

BokehModel(combine_events=True, render_bundle={'docs_json': {'722df8d5-6d24-473e-bf4e-17876b3d3a02': {'version…

In [32]:
def plot_bc_trenches_per_trench_unfold(kymo_imgs_unfold: xr.Dataset, 
                                       groupby_cols: list,
                                       channels: list,
                                       cmap='gray',
                                       rasterize=False) -> dict:
    '''
    Plots the barcode trench images per trenchid and timepoint
        kymo_imgs_unfold (xr.Dataset): the barcode kymograph images
        trenchid (int): the trenchid to plot
        channels (list): the list of channels to plot
    Returns:
        dict: a dictionary of holoviews images for each
    '''
    hv_kymos = {}
    for channel in channels:
        hv_kymos[channel] = \
            (kymo_imgs_unfold[channel]
             .hvplot.image(
                 x='x',
                 y='y',
                 groupby=groupby_cols,
                 cmap=cmap,
                 rasterize=rasterize,
                 data_aspect=1,
                 title=f'{channel} {trenchid}'
                 )
             .opts(invert_yaxis=False,
                   xaxis=None,
                   yaxis=None,
                   colorbar=False
                   )
            )
    return hv_kymos

In [33]:
groupby_cols

['File Trench Index', 'timepoints']

In [34]:
channels = ['BF', 'GFP', 'Cy5', 'Cy7']
groupby_cols = ['File Trench Index']
hv_kymos = plot_bc_trenches_per_trench_unfold(kymo_imgs_unfold, groupby_cols=groupby_cols, channels=channels)
hv_kymos['BF'] + hv_kymos['GFP'] + hv_kymos['Cy5'] + hv_kymos['Cy7']

BokehModel(combine_events=True, render_bundle={'docs_json': {'cb0014ba-8ec0-42c7-b3f8-7b36b77f5d99': {'version…

#### Kymographs for phenotype and segmentation

In [35]:
prefix = '/home/lag36/scratch/lag36/2024-12-05_lLAG10_MBM_Run-2_3-Fiducials_Temp-Fixed/Growth_Division/'
file_index = 100
filename = f'{prefix}/kymograph/kymograph_{file_index}.hdf5'

kymo_imgs = load_kymo_images(filename, df_bar_kymo_meta_trench, file_index=None)
kymo_imgs_unfold = unfold_xr_kymograph(kymo_imgs)

channels = ['mCherry']
groupby_cols = ['File Trench Index']
hv_kymos_phe = plot_bc_trenches_per_trench_unfold(kymo_imgs_unfold, groupby_cols=groupby_cols, channels=channels)

In [36]:
prefix = '/home/lag36/scratch/lag36/2024-12-05_lLAG10_MBM_Run-2_3-Fiducials_Temp-Fixed/Growth_Division/'
file_index = 100
filename = f'{prefix}/fluorsegmentation/segmentation_{file_index}.hdf5'

kymo_imgs = load_kymo_images(filename, df_bar_kymo_meta_trench, file_index=None)
kymo_imgs_unfold = unfold_xr_kymograph(kymo_imgs)

channels = ['data']
groupby_cols = ['File Trench Index']
hv_kymos_seg = plot_bc_trenches_per_trench_unfold(kymo_imgs_unfold, groupby_cols=groupby_cols, channels=channels, cmap='jet')

In [37]:
(hv_kymos_phe['mCherry'] + hv_kymos_seg['data']).cols(1)

BokehModel(combine_events=True, render_bundle={'docs_json': {'9c408bf7-0ad3-41b4-b229-1479e7950aef': {'version…

In [38]:
def map_bc_to_pheno(barcode: str,
                    cycleorder: list = [0,1,6,7,12,13,18,19,24,25,4,5,10,11,16,
                                  2,3,8,9,14,15,20,21,26,27,17,22,23,28,29])-> str:
    '''
    Converts barcode - as ordered by bit number in RNA sequence - to Barcode: As ordered according to FISH readout
        barcode (str): the barcode as ordered by bit number in RNA sequence
        cycleorder (list): The mapping from order in FISH readout to position in the RNA barcode sequence
        Returns:
            str: the barcode as ordered according to FISH readout
    '''
    barcode_phe = [-1]*30
    for i, bit in enumerate(barcode):
        # print(i, bit)
        # Find the position of i in cycleorder
        i_phe = cycleorder.index(i)
        barcode_phe[i_phe] = bit
    barcode_phe = ''.join(barcode_phe)
    return barcode_phe
    

#### Show entire FOVs

In [39]:
prefix = '/home/lag36/scratch/lag36/2024-12-05_lLAG10_MBM_Run-2_3-Fiducials_Temp-Fixed/'
headpath_bar_fovs = prefix + '/Barcodes/hdf5/hdf5_'
headpath_phe_fovs = prefix + '/Growth_Division/hdf5/hdf5_'

fov_file_index = 0

data_fov = (xr
        # TODO: Parametrize the file name
        .open_dataset(headpath_bar_fovs+str(fov_file_index)+'.hdf5', phony_dims='sort')
        .swap_dims({'phony_dim_0':'timepoints', 'phony_dim_1':'y', 'phony_dim_2':'x'})
)

data_fov = (data_fov
        .assign_coords({
                        'timepoints': np.arange(data_fov.sizes['timepoints'], dtype=np.int16),
                        'y': np.arange(data_fov.sizes['y'], dtype=np.int16),
                        'x': np.arange(data_fov.sizes['x'], dtype=np.int16),})
)

In [40]:
channels = ['BF', 'RFP', 'Cy5']
hv_fovs = {}
for channel in channels:
    hv_fovs[channel] = (data_fov[channel]
                .hvplot.image(
                    x='x',
                    y='y',
                    cmap='gray',
                    data_aspect=1,
                    title=channel,
                    rasterize=True
                )
                .opts(invert_yaxis=True,
                    xaxis=None,
                    yaxis=None,
                    colorbar=False)
    )

In [41]:
hv_fovs['Cy5'] + hv_fovs['RFP']

BokehModel(combine_events=True, render_bundle={'docs_json': {'0cf8910c-d232-46b6-8a03-fda4836b0abe': {'version…

In [98]:
dask_controller_instance.shutdown()

Done.
